In [ ]:
# import urllib.request
# import os

# # 先把 download_reference_model.sh 里的链接填进来
# url = "https://www.dropbox.com/scl/fi/ng66ceortfy07bgie3r54/alltracker_reference.tar.gz?rlkey=o781im2v0sl7035hy8fcuv1d5&st=u5mcttcx&dl=1"
# save_path = "./alltracker_reference.tar.gz"


# urllib.request.urlretrieve(url, save_path)

# size = os.path.getsize(save_path)
# print(f" 下载完成，文件大小: {size/1024/1024:.1f} MB")

In [17]:
import sys


import torch
import numpy as np
import cv2
import json
import os
sys.path.insert(0, './alltracker')  # 从外层目录指向 alltracker/

from nets.alltracker import Net as AllTracker


# ============================================================
# 加载模型
# ============================================================
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = AllTracker(seqlen=16).to(device)
# 自动从 huggingface 下载权重（第一次运行需要联网）
checkpoint = torch.load(r'.\EndoTracker\checkpoints\reference_model\model-000600000.pth', map_location=device)
print(checkpoint.keys())

print(type(checkpoint))
print(checkpoint.keys())
model.load_state_dict(checkpoint['model'])
model.eval()
print("✅ AllTracker 模型加载完成")

# ============================================================
# 读取视频（和 CoTracker 相同）
# ============================================================
def read_video_for_tracker(mp4_path, max_frames=50):
    cap = cv2.VideoCapture(mp4_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"无法打开视频: {mp4_path}")
    frames = []
    while len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    print(f"读取 {len(frames)} 帧，分辨率 {frames[0].shape}")
    return torch.tensor(np.array(frames)).permute(0,3,1,2).unsqueeze(0).float()

video_path = "./EndoTracker/alltracker//dataset/dynamic_seq_003.mp4"
MAX_FRAMES  =16
video_tensor = read_video_for_tracker(video_path, max_frames=MAX_FRAMES).to(device)
B, T, C, H, W = video_tensor.shape


with open("./EndoTracker/annotation_003.json", "r") as f:
    ann = json.load(f)
f0_pts = ann["f0"]    
fL_pts = ann["fLast"]
N = len(f0_pts)



dict_keys(['model'])
<class 'dict'>
dict_keys(['model'])
✅ AllTracker 模型加载完成
读取 16 帧，分辨率 (480, 480, 3)


In [ ]:
# %cd alltracker/
# with torch.no_grad():
#     output = model(video_tensor, iters=4)

# flow_field = output[0]  # [1, 16, 2, 480, 480]  (B, T, 2, H, W)
# conf_field = output[1]  # [1, 16, 2, 480, 480]  置信度

# print(f"flow_field shape: {flow_field.shape}")

# # ============================================================
# # 从流场中采样标注点的轨迹
# # ============================================================
# def sample_tracks_from_flow(flow_field, query_points, H, W):
#     """
#     flow_field:   [B, T, 2, H, W]  密集流场，每像素的位移
#     query_points: [[x1,y1], ...]   第一帧标注点坐标
#     返回:
#         trajs: [B, T, N, 2]  每个标注点在各帧的绝对坐标
#         confs: [B, T, N]     置信度
#     """
#     B, T, _, fH, fW = flow_field.shape
#     N = len(query_points)

#     trajs = torch.zeros(B, T, N, 2)
#     confs = torch.zeros(B, T, N)

#     for i, (x, y) in enumerate(query_points):
#         x_idx = int(round(x * fW / W))  # 缩放到流场分辨率
#         y_idx = int(round(y * fH / H))

#         # 确保不越界
#         x_idx = min(max(x_idx, 0), fW - 1)
#         y_idx = min(max(y_idx, 0), fH - 1)

#         # flow_field[:, :, 0, y, x] 是 x 方向位移
#         # flow_field[:, :, 1, y, x] 是 y 方向位移
#         dx = flow_field[0, :, 0, y_idx, x_idx].cpu()  # [T]
#         dy = flow_field[0, :, 1, y_idx, x_idx].cpu()  # [T]

#         # 绝对坐标 = 原始坐标 + 位移
#         trajs[0, :, i, 0] = x + dx
#         trajs[0, :, i, 1] = y + dy

#         # 置信度取两个通道的均值
#         confs[0, :, i] = conf_field[0, :, 0, y_idx, x_idx].cpu()

#     return trajs, confs

# B, T, C, H, W = video_tensor.shape
# trajs, confs = sample_tracks_from_flow(flow_field, f0_pts, H, W)

# # 可见性：置信度 > 0 视为可见
# vis_scores = (confs > 0.1).float()

# print(f"trajs shape: {trajs.shape}")   # [1, T, N, 2]
# print(f"confs shape: {confs.shape}")   # [1, T, N]

# # ============================================================
# # 可视化
# # ============================================================
# import sys
# sys.path.insert(0, '../co-tracker')
# from cotracker.utils.visualizer import Visualizer
# import os

# os.makedirs("../vis_output", exist_ok=True)
# vis_tool = Visualizer(save_dir="../vis_output", pad_value=120, linewidth=2, fps=10)
# vis_tool.visualize(
#     video=video_tensor.cpu(),
#     tracks=trajs,
#     visibility=vis_scores,
#     filename="dynamic_seq_003_alltracker"
# )
# print("✅ 轨迹视频 → ../vis_output/dynamic_seq_003_alltracker.mp4")

# # ============================================================
# # 评估
# # ============================================================
# pred_last     = trajs[0, -1].numpy()        # [N, 2]
# gt_last       = np.array(fL_pts, dtype=np.float32)
# pred_vis_last = vis_scores[0, -1].numpy()   # [N]

# print("\n各点置信度（最后一帧）:")
# for i, (c, v) in enumerate(zip(confs[0, -1].numpy(), pred_vis_last)):
#     print(f"  点{i+1}: conf={c:.3f}  {'✅' if v > 0.5 else '❌'}")

# errors = np.linalg.norm(pred_last - gt_last, axis=1)
# print(f"\n{'='*50}")
# print(f"  AllTracker 评估结果")
# print(f"{'='*50}")
# print(f"ATE:       {errors.mean():.2f} px")
# print(f"中位误差:  {np.median(errors):.2f} px")
# for i, e in enumerate(errors):
#     print(f"  点{i+1}: {e:.2f}px  {'✅' if e < 10 else '❌'}")

# # 最后一帧对比图
# last_frame_vis = video_tensor[0, -1].permute(1,2,0).cpu().numpy().astype(np.uint8)
# last_frame_vis = cv2.cvtColor(last_frame_vis, cv2.COLOR_RGB2BGR)
# for i, ((px,py),(gx,gy)) in enumerate(zip(pred_last, gt_last)):
#     cv2.circle(last_frame_vis, (int(gx),int(gy)), 6, (0,255,0), 2)
#     cv2.circle(last_frame_vis, (int(px),int(py)), 4, (0,0,255), -1)
#     cv2.line(last_frame_vis, (int(gx),int(gy)), (int(px),int(py)), (0,255,255), 1)
#     cv2.putText(last_frame_vis, f"P{i+1}", (int(px)+6,int(py)-6),
#                 cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0,0,255), 1)
# cv2.imwrite("../vis_output/alltracker_last_frame.png", last_frame_vis)
# print("✅ 对比图 → ../vis_output/alltracker_last_frame.png")


In [18]:
def track_long_video(model, video_tensor, f0_pts, window_size=16, stride=8):
    """
    用滑窗方式追踪完整视频
    window_size: 每次处理的帧数
    stride:      窗口滑动步长（overlap = window_size - stride）
    """
    B, T, C, H, W = video_tensor.shape
    N = len(f0_pts)
    device = video_tensor.device

    # 存储每个点在所有帧的坐标
    all_trajs = torch.zeros(B, T, N, 2)
    all_confs = torch.zeros(B, T, N)

    # 第0帧的初始坐标
    for i, (x, y) in enumerate(f0_pts):
        all_trajs[0, 0, i, 0] = x
        all_trajs[0, 0, i, 1] = y

    # 滑窗推理
    for start in range(0, T - 1, stride):
        end = min(start + window_size, T)
        chunk = video_tensor[:, start:end].to(device)  # [1, W, C, H, W]

        # 如果帧数不足 window_size，padding 补齐
        chunk_len = chunk.shape[1]
        if chunk_len < window_size:
            pad = video_tensor[:, -1:].repeat(1, window_size - chunk_len, 1, 1, 1)
            chunk = torch.cat([chunk, pad], dim=1)

        with torch.no_grad():
            output = model(chunk, iters=4)

        flow_field = output[0]  # [1, 16, 2, H, W]
        conf_field = output[1]  # [1, 16, 2, H, W]
        _, fT, _, fH, fW = flow_field.shape

        # 取这个窗口内有效帧的轨迹
        for i, (x0, y0) in enumerate(f0_pts):
            # 用窗口第一帧对应的实际坐标作为查询点
            if start == 0:
                qx, qy = x0, y0
            else:
                # 用上一窗口预测的坐标继续追踪
                qx = all_trajs[0, start, i, 0].item()
                qy = all_trajs[0, start, i, 1].item()

            x_idx = min(max(int(round(qx * fW / W)), 0), fW - 1)
            y_idx = min(max(int(round(qy * fH / H)), 0), fH - 1)

            for t_local in range(1, chunk_len):
                t_global = start + t_local
                if t_global >= T:
                    break
                dx = flow_field[0, t_local, 0, y_idx, x_idx].cpu().item()
                dy = flow_field[0, t_local, 1, y_idx, x_idx].cpu().item()
                all_trajs[0, t_global, i, 0] = qx + dx
                all_trajs[0, t_global, i, 1] = qy + dy
                all_confs[0, t_global, i]     = conf_field[0, t_local, 0, y_idx, x_idx].cpu().item()

        print(f"  处理帧 {start}~{end-1} / {T-1}")

    return all_trajs, all_confs

In [19]:
def alltracker_metrics(pred_trajs, gt_trajs, pred_vis, gt_vis, H, W):
    """
    AllTracker 官方三指标
    pred_trajs: [T, N, 2] 预测轨迹
    gt_trajs:   [T, N, 2] GT 轨迹（你只有第0帧和最后帧，中间帧用线性插值）
    pred_vis:   [T, N]    预测可见性
    gt_vis:     [T, N]    GT 可见性（简化：全1）
    """
    T, N, _ = pred_trajs.shape

    # ── 1. d_avg：不同阈值下的追踪精度（只看 GT 可见的点）──
    diag = np.sqrt(H**2 + W**2)
    thresholds = np.array([1, 2, 4, 8, 16]) / 256.0 * diag

    vis_mask = gt_vis > 0.5  # [T, N]
    errors = np.linalg.norm(pred_trajs - gt_trajs, axis=-1)  # [T, N]

    accs = []
    for thresh in thresholds:
        correct = (errors < thresh) & vis_mask
        acc = correct.sum() / (vis_mask.sum() + 1e-6)
        accs.append(acc)
    d_avg = np.mean(accs)

    # ── 2. Occlusion Accuracy（OA）：可见性预测准确率 ──
    pred_vis_bin = (pred_vis > 0.5).astype(float)
    gt_vis_bin   = (gt_vis > 0.5).astype(float)
    OA = (pred_vis_bin == gt_vis_bin).mean()

    # ── 3. Average Jaccard（AJ）：同时考虑位置精度和可见性 ──
    AJs = []
    for thresh in thresholds:
        within = (errors < thresh)  # [T, N]
        # True Positive: 预测可见 & GT可见 & 位置正确
        TP = (pred_vis_bin * gt_vis_bin * within).sum()
        # False Positive: 预测可见 但 GT不可见 或 位置错误
        FP = (pred_vis_bin * (1 - gt_vis_bin * within)).sum()
        # False Negative: GT可见 但 预测不可见 或 位置错误
        FN = (gt_vis_bin * (1 - pred_vis_bin * within)).sum()
        AJ = TP / (TP + FP + FN + 1e-6)
        AJs.append(AJ)
    avg_jaccard = np.mean(AJs)

    return {
        "d_avg": d_avg,
        "AJ":    avg_jaccard,
        "OA":    OA,
        "per_threshold": dict(zip([1,2,4,8,16], accs))
    }

In [21]:
# 读取完整视频（不限帧数）
MAX_FRAMES = 50  # 或者视频全长
%cd ..
video_tensor = read_video_for_tracker(video_path, max_frames=MAX_FRAMES).to(device)
B, T, C, H, W = video_tensor.shape

# 滑窗追踪
print("开始滑窗追踪...")
trajs, confs = track_long_video(model, video_tensor, f0_pts,
                                 window_size=16, stride=8)

# 构造 GT（只有第0帧和最后帧，中间帧线性插值）
gt_trajs = np.zeros((T, len(f0_pts), 2), dtype=np.float32)
for i, ((x0,y0), (xL,yL)) in enumerate(zip(f0_pts, fL_pts)):
    gt_trajs[:, i, 0] = np.linspace(x0, xL, T)
    gt_trajs[:, i, 1] = np.linspace(y0, yL, T)
gt_vis = np.ones((T, len(f0_pts)))  # 假设全程可见

# 评估
pred_trajs_np = trajs[0].numpy()   # [T, N, 2]
pred_vis_np   = confs[0].numpy()   # [T, N]

results = alltracker_metrics(pred_trajs_np, gt_trajs, pred_vis_np, gt_vis, H, W)

print(f"\n{'='*45}")
print(f"      AllTracker 官方指标")
print(f"{'='*45}")
print(f"d_avg (追踪精度):    {results['d_avg']*100:.1f}%")
print(f"AJ    (平均Jaccard): {results['AJ']*100:.1f}%")
print(f"OA    (遮挡精度):    {results['OA']*100:.1f}%")
print("-" * 45)
for thresh, acc in results["per_threshold"].items():
    bar = "█" * int(acc * 20)
    print(f"  δ<{thresh:>2}px: {acc*100:5.1f}%  {bar}")

d:\
读取 50 帧，分辨率 (480, 480, 3)
开始滑窗追踪...
  处理帧 0~15 / 49
  处理帧 8~23 / 49
  处理帧 16~31 / 49
  处理帧 24~39 / 49
  处理帧 32~47 / 49
  处理帧 40~49 / 49
  处理帧 48~49 / 49

      AllTracker 官方指标
d_avg (追踪精度):    16.5%
AJ    (平均Jaccard): 8.7%
OA    (遮挡精度):    90.4%
---------------------------------------------
  δ< 1px:   4.0%  
  δ< 2px:   7.6%  █
  δ< 4px:  12.0%  ██
  δ< 8px:  22.8%  ████
  δ<16px:  36.0%  ███████


In [22]:
import cv2
import os
import numpy as np

def save_tracking_video(video_tensor, trajs, confs, output_path, fps=10, conf_thr=0.1):
    """
    将追踪结果渲染成视频
    video_tensor: [1, T, C, H, W]
    trajs:        [1, T, N, 2]
    confs:        [1, T, N]
    """
    B, T, C, H, W = video_tensor.shape
    N = trajs.shape[2]

    # 每个点分配一个固定颜色
    colors = [
        (0, 0, 255),    # 红
        (0, 255, 0),    # 绿
        (255, 0, 0),    # 蓝
        (0, 255, 255),  # 黄
        (255, 0, 255),  # 紫
    ]

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(output_path, fourcc, fps, (W, H))

    trajs_np = trajs[0].numpy()   # [T, N, 2]
    confs_np = confs[0].numpy()   # [T, N]

    for t in range(T):
        # 取当前帧图像
        frame = video_tensor[0, t].permute(1, 2, 0).cpu().numpy().astype(np.uint8)
        frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

        for i in range(N):
            color = colors[i % len(colors)]
            conf  = confs_np[t, i]
            px    = int(round(trajs_np[t, i, 0]))
            py    = int(round(trajs_np[t, i, 1]))

            # 越界跳过
            if not (0 <= px < W and 0 <= py < H):
                continue

            if conf > conf_thr:
                # 画当前点
                cv2.circle(frame, (px, py), 5, color, -1)
                # 画轨迹线（往前追溯 trail_len 帧）
                trail_len = 20
                for t_prev in range(max(0, t - trail_len), t):
                    if confs_np[t_prev, i] <= conf_thr:
                        continue
                    px_prev = int(round(trajs_np[t_prev, i, 0]))
                    py_prev = int(round(trajs_np[t_prev, i, 1]))
                    if not (0 <= px_prev < W and 0 <= py_prev < H):
                        continue
                    # 越老的轨迹越透明（颜色变暗）
                    alpha = (t_prev - (t - trail_len)) / trail_len
                    faded = tuple(int(c * alpha) for c in color)
                    cv2.line(frame, (px_prev, py_prev), (px, py), faded, 1)

                # 点编号标注
                cv2.putText(frame, f"P{i+1}", (px+6, py-6),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
            else:
                # 不可见：画空心圆表示追踪失效
                cv2.circle(frame, (px, py), 5, color, 1)
                cv2.putText(frame, f"P{i+1}?", (px+6, py-6),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (128,128,128), 1)

        # 帧号
        cv2.putText(frame, f"Frame {t+1}/{T}", (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)

        writer.write(frame)

    writer.release()
    print(f"✅ 视频已保存 → {output_path}")


# ============================================================
# 调用
# ============================================================
save_tracking_video(
    video_tensor=video_tensor.cpu(),
    trajs=trajs,
    confs=confs,
    output_path="./vis_output/alltracker_full_video.mp4",
    fps=10,
    conf_thr=0.1
)

✅ 视频已保存 → ./vis_output/alltracker_full_video.mp4


In [23]:
%cd alltracker/
from utils.improc import flow2color  # AllTracker 自带的工具

with torch.no_grad():
    output = model(video_tensor, iters=4)

flow_field = output[0]  # [1, T, 2, H, W]

# 可视化每一帧的流场
import os
os.makedirs("../vis_output/flow_frames", exist_ok=True)

for t in range(flow_field.shape[1]):
    # 取第t帧的流场 [2, H, W]
    flow_t = flow_field[0, t]  # [2, H, W]
    
    # 转成彩色图（官方方式）
    flow_rgb = flow2color(flow_t.unsqueeze(0))  # 颜色代表运动方向和大小
    
    frame = flow_rgb[0].permute(1,2,0).cpu().numpy().astype(np.uint8)
    frame_bgr = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
    cv2.imwrite(f"../vis_output/flow_frames/frame_{t:03d}.png", frame_bgr)

print("✅ 光流图已保存")
%cd ..

[WinError 2] 系统找不到指定的文件。: 'alltracker/'
d:\
✅ 光流图已保存
d:\
